In [ ]:
!pip install pyspark imbalanced-learn

In [ ]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, when

from pyspark.ml.stat import Correlation
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)
from pyspark.ml.clustering import KMeans
from imblearn.over_sampling import ADASYN

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

spark = SparkSession.builder.getOrCreate()

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')
COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']


## 1. Data Loading

In [ ]:
# Load all the csv files, use inferSchema to auto detect column types
customer          = spark.read.csv('data/olist_customers_dataset.csv',          header=True, inferSchema=True)
geolocation       = spark.read.csv('data/olist_geolocation_dataset.csv',        header=True, inferSchema=True)
items             = spark.read.csv('data/olist_order_items_dataset.csv',         header=True, inferSchema=True)
payments          = spark.read.csv('data/olist_order_payments_dataset.csv',      header=True, inferSchema=True)
review            = spark.read.csv('data/olist_order_reviews_dataset.csv',       header=True, inferSchema=True)
orders            = spark.read.csv('data/olist_orders_dataset.csv',              header=True, inferSchema=True)
product           = spark.read.csv('data/olist_products_dataset.csv',            header=True, inferSchema=True)
seller            = spark.read.csv('data/olist_sellers_dataset.csv',             header=True, inferSchema=True)
prodCat_translation = spark.read.csv('data/product_category_name_translation.csv', header=True, inferSchema=True)


## 2. Data Cleaning

In [ ]:
# GEOLOCATION: multiple lat/lng rows per zip code, collapse to one row per zip by averaging coordinates
geolocation_clean = geolocation.groupBy('geolocation_zip_code_prefix').agg(
    F.avg('geolocation_lat').alias('geo_lat'),
    F.avg('geolocation_lng').alias('geo_lng'),
    F.first('geolocation_city').alias('geo_city'),
    F.first('geolocation_state').alias('geo_state')
)

# PAYMENTS: one order can have multiple payment types/rows (e.g. voucher + credit card)
# Sum all into a single total_order_value per order
payments_clean = payments.groupBy('order_id').agg(
    F.sum('payment_value').alias('total_order_value'),
    F.max('payment_installments').alias('max_installments')
)

# ORDERS: date columns are stored as strings, change to Timestamp
orders_clean = orders \
    .withColumn('order_purchase_timestamp',      F.to_timestamp('order_purchase_timestamp')) \
    .withColumn('order_delivered_customer_date', F.to_timestamp('order_delivered_customer_date')) \
    .withColumn('order_estimated_delivery_date', F.to_timestamp('order_estimated_delivery_date')) \
    .dropDuplicates(['order_id'])

# REVIEWS: take the highest score if an order has multiple reviews
review_clean = review.groupBy('order_id').agg(
    F.max('review_score').alias('review_score'),
    F.first('review_creation_date').alias('review_date')
)

# PRODUCTS: trim whitespace from category name so the join with translations works correctly
product_clean = product \
    .withColumn('product_category_name', F.trim(F.col('product_category_name'))) \
    .fillna({'product_category_name': 'unclassified'})

prodCat_translation_clean = prodCat_translation \
    .withColumn('product_category_name', F.trim(F.col('product_category_name')))

# ITEMS: cast price and freight to double explicitly
items_clean = items \
    .withColumn('price',         F.col('price').cast('double')) \
    .withColumn('freight_value', F.col('freight_value').cast('double'))

# CUSTOMERS / SELLERS: deduplicate on primary key
customer_clean = customer.dropDuplicates(['customer_id'])
seller_clean   = seller.dropDuplicates(['seller_id'])

In [ ]:
# Register all cleaned tables as temporary SQL views
orders_clean.createOrReplaceTempView('orders')
payments_clean.createOrReplaceTempView('payments')
review_clean.createOrReplaceTempView('reviews')
items_clean.createOrReplaceTempView('items')
product_clean.createOrReplaceTempView('products')
prodCat_translation_clean.createOrReplaceTempView('translations')
customer_clean.createOrReplaceTempView('customers')
seller_clean.createOrReplaceTempView('sellers')


## 3. Data Integration (Join)

In [ ]:
# Join all tables into 1 dataframe
# Use inner join on orders/items/products/customers/sellers so we only keep rows where all columns exist.
# LEFT JOIN on payments/reviews/translation keeps orders even if a payment or review is missing.
query = """
SELECT
    r.review_score          AS label,
    o.order_id,
    o.customer_id,
    i.product_id,
    s.seller_id,

    -- numerical features
    i.price,
    i.freight_value,
    p.total_order_value,
    p.max_installments,
    pr.product_weight_g,
    pr.product_photos_qty,

    -- categorical features
    t.product_category_name_english AS category,
    c.customer_state,
    s.seller_state,

    -- timestamps needed for delivery feature engineering
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date

FROM orders o
LEFT  JOIN payments p    ON o.order_id   = p.order_id
LEFT  JOIN reviews  r    ON o.order_id   = r.order_id
INNER JOIN items    i    ON o.order_id   = i.order_id
INNER JOIN products pr   ON i.product_id = pr.product_id
LEFT  JOIN translations t ON pr.product_category_name = t.product_category_name
INNER JOIN customers c   ON o.customer_id = c.customer_id
INNER JOIN sellers   s   ON i.seller_id   = s.seller_id
"""

df = spark.sql(query)

# Each row = one ordered item with its associated order, customer, seller, review + payment info
df.show(5)


+-----+--------------------+--------------------+--------------------+--------------------+-----+-------------+-----------------+----------------+----------------+------------------+---------------+--------------+------------+------------------------+-----------------------------+-----------------------------+
|label|            order_id|         customer_id|          product_id|           seller_id|price|freight_value|total_order_value|max_installments|product_weight_g|product_photos_qty|       category|customer_state|seller_state|order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|
+-----+--------------------+--------------------+--------------------+--------------------+-----+-------------+-----------------+----------------+----------------+------------------+---------------+--------------+------------+------------------------+-----------------------------+-----------------------------+
|    4|00018f77f2f0320c5...|f6dd3ec061db4e398...|e5f2d52b802189e

## 4. Null Check & Imputation

In [ ]:
# Count nulls per column to find which fields to drop / imputation
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()


+-----+--------+-----------+----------+---------+-----+-------------+-----------------+----------------+----------------+------------------+--------+--------------+------------+------------------------+-----------------------------+-----------------------------+
|label|order_id|customer_id|product_id|seller_id|price|freight_value|total_order_value|max_installments|product_weight_g|product_photos_qty|category|customer_state|seller_state|order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|
+-----+--------+-----------+----------+---------+-----+-------------+-----------------+----------------+----------------+------------------+--------+--------------+------------+------------------------+-----------------------------+-----------------------------+
|  942|       0|          0|         0|        0|    0|            0|                3|               3|              18|              1603|    1627|             0|           0|                       0|         

In [ ]:
# Drop rows where the target (label) are missing bcs these rows can't be used for modelling
# Fill missing product attributes with 0 and unknown category
df_clean = df.dropna(subset=['label', 'total_order_value', 'max_installments', 'order_delivered_customer_date'])
df_clean = df_clean.fillna({'product_weight_g': 0, 'product_photos_qty': 0, 'category': 'unknown'})


## 5. Feature Engineering

In [ ]:
# Derive two delivery related features from the timestamp columns:
# delivery_days= total transit time from purchase to actual delivery
# delivery_error = how many days late (positive) or early (negative) vs the estimate
# unix_timestamp converts a Timestamp to seconds; dividing by 86400 converts to days.
df_clean = df_clean.withColumn(
    'delivery_days',
    (F.unix_timestamp('order_delivered_customer_date') -
     F.unix_timestamp('order_purchase_timestamp')) / 86400
)

df_clean = df_clean.withColumn(
    'delivery_error',
    (F.unix_timestamp('order_delivered_customer_date') -
     F.unix_timestamp('order_estimated_delivery_date')) / 86400
)

df_clean = df_clean.fillna({'category': 'unknown'})

In [ ]:
# Check again no nulls remain
df_clean.select([count(when(col(c).isNull(), c)).alias(c) for c in df_clean.columns]).show()


+-----+--------+-----------+----------+---------+-----+-------------+-----------------+----------------+----------------+------------------+--------+--------------+------------+------------------------+-----------------------------+-----------------------------+-------------+--------------+
|label|order_id|customer_id|product_id|seller_id|price|freight_value|total_order_value|max_installments|product_weight_g|product_photos_qty|category|customer_state|seller_state|order_purchase_timestamp|order_delivered_customer_date|order_estimated_delivery_date|delivery_days|delivery_error|
+-----+--------+-----------+----------+---------+-----+-------------+-----------------+----------------+----------------+------------------+--------+--------------+------------+------------------------+-----------------------------+-----------------------------+-------------+--------------+
|    0|       0|          0|         0|        0|    0|            0|                0|               0|               0|   

In [ ]:
# Cast all numerical columns to double
# This is because Spark requires numeric type to be consistent (mixing int and double can cause pipeline errors)
df_clean = df_clean \
    .withColumn('label',             col('label').cast('double')) \
    .withColumn('price',             col('price').cast('double')) \
    .withColumn('freight_value',     col('freight_value').cast('double')) \
    .withColumn('total_order_value', col('total_order_value').cast('double')) \
    .withColumn('product_weight_g',  col('product_weight_g').cast('double')) \
    .withColumn('product_photos_qty',col('product_photos_qty').cast('double'))

# Remove any duplicate orders
df_clean = df_clean.dropDuplicates(['order_id'])

print(f'Rows after cleaning: {df_clean.count():,}')


Rows after cleaning: 95,829


## 6. Outlier Treatment (Winsorisation)

In [ ]:
# Rather than removing outlier rows, we use winsorisation (cap extreme values at the 1st and 99th percentile)
# This will help us keep all rows while preventing extreme values from distorting the model (bcs there's just too many rows to drop if we remove all outliers)
outlier_cols = ['total_order_value', 'delivery_days', 'delivery_error', 'product_weight_g']

for col_name in outlier_cols:
    # approxQuantile returns [p01, p99] with a relative error of 0.01
    p01, p99 = df_clean.approxQuantile(col_name, [0.01, 0.99], 0.01)
    # Clip values below p1 up to p1, and values above p99 down to p99
    df_clean = df_clean.withColumn(
        col_name,
        F.when(col(col_name) < p01, p01)
         .when(col(col_name) > p99, p99)
         .otherwise(col(col_name))
    )
    print(f'{col_name}: capped at [{p01:.2f}, {p99:.2f}]')

print(f'Rows after winsorisation: {df_clean.count():,}  (no rows dropped)')
# IF using IQR, rows are dropped into 65k

total_order_value: capped at [9.59, 13664.08]
delivery_days: capped at [0.53, 208.35]
delivery_error: capped at [-146.02, 188.98]
product_weight_g: capped at [0.00, 40425.00]
Rows after winsorisation: 95,829  (no rows dropped)


## 7. Processed Dataset Output

In [ ]:
# Select the final feature columns and save the processed dataset
# Parquet for the standard Spark output format
# CSV for easy to read / inspection
output_cols = [
    'order_id', 'customer_id', 'label',
    'price', 'freight_value', 'total_order_value', 'max_installments',
    'product_weight_g', 'product_photos_qty',
    'category', 'customer_state', 'seller_state',
    'delivery_days', 'delivery_error'
]

df_output = df_clean.select(output_cols)

df_output.write.mode('overwrite').parquet('olist_processed.parquet')
print('Parquet saved to olist_processed.parquet')

# coalesce(1) writes a single CSV file instead of one file per Spark partition
df_output.coalesce(1).write.mode('overwrite').option('header', True).csv('olist_processed_csv')
print('CSV saved to olist_processed_csv/')

print(f'Final dataset: {df_output.count():,} rows x {len(output_cols)} columns')
df_output.printSchema()


Parquet saved to olist_processed.parquet
CSV saved to olist_processed_csv/
Final dataset: 95,829 rows x 14 columns
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- label: double (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- total_order_value: double (nullable = true)
 |-- max_installments: integer (nullable = true)
 |-- product_weight_g: double (nullable = false)
 |-- product_photos_qty: double (nullable = false)
 |-- category: string (nullable = false)
 |-- customer_state: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- delivery_days: double (nullable = true)
 |-- delivery_error: double (nullable = true)



## 8. Feature Selection

In [ ]:
# Pearson correlation between each numerical feature and the review score (label)
# Scores near 0 means little linear relationship
# Scores near +/-1 means strong relationship
# This helps us confirm which features are worth including in the model
num_features = ['price', 'freight_value', 'total_order_value', 'product_weight_g', 'product_photos_qty']

assembler_corr = VectorAssembler(inputCols=num_features + ['label'], outputCol='corr_features', handleInvalid='skip')
df_vector = assembler_corr.transform(df_clean)

matrix = Correlation.corr(df_vector, 'corr_features').collect()[0][0]
corr_matrix = matrix.toArray().tolist()

print('Pearson correlation with review score:')
for i, feature in enumerate(num_features):
    print(f'  {feature:<25} {corr_matrix[i][-1]:+.4f}')


Pearson correlation with review score:
  price                     -0.0058
  freight_value             -0.0354
  total_order_value         -0.0422
  product_weight_g          -0.0288
  product_photos_qty        +0.0140


In [ ]:
# Group-mean analysis for categorical features
# If average review score differs meaningfully across categories or states,
# those columns carry signal and are worth including as encoded features

# Categories with at least 100 orders, ranked by average score
print('Avg review score by product category (top 10):')
df_clean.groupBy('category').agg(
    F.avg('label').alias('avg_score'),
    F.stddev('label').alias('std_dev'),
    F.count('label').alias('count')
).filter('count > 100').orderBy(F.desc('avg_score')).show(10)

print('Avg review score by customer state:')
df_clean.groupBy('customer_state').agg(
    F.avg('label').alias('avg_score'),
    F.count('label').alias('count')
).orderBy('avg_score').show()


Avg review score by product category (top 10):
+--------------------+------------------+------------------+-----+
|            category|         avg_score|           std_dev|count|
+--------------------+------------------+------------------+-----+
|books_general_int...| 4.538934426229508|0.9982122067247478|  488|
|          food_drink|4.4495412844036695|0.9355866444405272|  218|
|     books_technical| 4.425196850393701|1.1352747576362334|  254|
| luggage_accessories| 4.379482071713148|1.0894295573354444| 1004|
|                food| 4.324825986078887|1.1856941465587232|  431|
|          stationery| 4.306624888093107|1.1612033290359547| 2234|
|       fashion_shoes| 4.296943231441048| 1.188039554730945|  229|
|            pet_shop| 4.280167264038232|1.2011156283387583| 1674|
|costruction_tools...| 4.278688524590164|1.2376574941314253|  183|
|    small_appliances| 4.267886855241264|1.2273753951054505|  601|
+--------------------+------------------+------------------+-----+
only showing to

## 9. Classification — Customer Satisfaction Prediction

In [ ]:
# Final feature set selected based on correlation analysis and domain reasoning
# delivery_days and delivery_error are the strongest signals
# Categorical features (category, state) are included after being encoded to numeric indices
feature_columns = [
    'total_order_value',
    'delivery_days',
    'delivery_error',
    'freight_value',
    'product_weight_g',
    'category_index',
    'cust_state_index',
    'seller_state_index'
]

# StringIndexer converts string categories to integer indices ranked by frequency
# handleInvalid='keep' to assign unseen values their own index rather than crashing
indexer = StringIndexer(
    inputCols=['category', 'customer_state', 'seller_state'],
    outputCols=['category_index', 'cust_state_index', 'seller_state_index'],
    handleInvalid='keep'
)

In [ ]:
# Binary target: Satisfied = 4-5 stars (1.0), Dissatisfied = 1-3 stars (0.0)
df_clf = df_clean.withColumn('satisfaction', F.when(F.col('label') >= 4, 1.0).otherwise(0.0))

print('Label distribution before ADASYN:')
df_clf.groupBy('satisfaction').count().orderBy('satisfaction').show()

# 1. Transform categorical variables first
df_clf_indexed = indexer.fit(df_clf).transform(df_clf)

# 2. Random Split
train_clf, test_clf = df_clf_indexed.randomSplit([0.8, 0.2], seed=42)

# 3. Convert Train data to Pandas
train_pdf = train_clf.select(feature_columns + ['satisfaction']).toPandas()

# Split into continuous (safe to interpolate) and categorical (must NOT interpolate) because we should NOT interpolate categorical ones, since it cna cause
# averaging encoded labels (e.g. 3 and 7 becoming 5.3) creates fake categories that don't actually exist and can wrongly affect feature importance
# for example i forgot to split this before and suddenly cust_state_index, seller_steate_index and category_index importance got weirdly way too high
continuous_cols  = ['total_order_value', 'delivery_days', 'delivery_error',
                    'freight_value', 'product_weight_g']
categorical_cols = ['category_index', 'cust_state_index', 'seller_state_index']

X_cont  = train_pdf[continuous_cols]
X_cat   = train_pdf[categorical_cols]
y_train = train_pdf['satisfaction']

# 4. Apply ADASYN on continuous features only
adasyn = ADASYN(random_state=42)
X_cont_resampled, y_resampled = adasyn.fit_resample(X_cont, y_train)

n_original  = len(train_pdf)
n_synthetic = len(X_cont_resampled) - n_original
print(f'Training samples before ADASYN : {n_original:,}')
print(f'Synthetic samples generated    : {n_synthetic:,}')
print(f'Training samples after  ADASYN : {len(X_cont_resampled):,}')

# 5. Re-attach categorical columns safely
minority_mask = (y_train == 0.0)
minority_cat  = X_cat[minority_mask.values]

synthetic_cat = minority_cat.sample(
    n=n_synthetic, replace=True, random_state=42
).reset_index(drop=True)

X_cat_resampled = pd.concat(
    [X_cat.reset_index(drop=True), synthetic_cat],
    axis=0
).reset_index(drop=True)

# 6. Rebuild full feature DataFrame and convert back to Spark
X_resampled_full = pd.concat(
    [pd.DataFrame(X_cont_resampled, columns=continuous_cols), X_cat_resampled],
    axis=1
)

resampled_pdf = pd.concat(
    [X_resampled_full, y_resampled.reset_index(drop=True)],
    axis=1
)

train_clf_balanced = spark.createDataFrame(resampled_pdf)

Label distribution before ADASYN:
+------------+-----+
|satisfaction|count|
+------------+-----+
|         0.0|20128|
|         1.0|75701|
+------------+-----+

Training samples before ADASYN : 76,781
Synthetic samples generated    : 44,839
Training samples after  ADASYN : 121,620


In [ ]:
# 7. Apply VectorAssembler on balanced train data and original test data
assembler = VectorAssembler(inputCols=feature_columns, outputCol='features', handleInvalid='skip')
train_assembled = assembler.transform(train_clf_balanced)
test_assembled  = assembler.transform(test_clf)

# 8. Setup RandomForest and Hyperparameter Grid
rf_clf = RandomForestClassifier(labelCol='satisfaction', featuresCol='features', seed=42)

paramGrid = (ParamGridBuilder()
    .addGrid(rf_clf.numTrees, [50, 100])
    .addGrid(rf_clf.maxDepth, [5, 10])
    .build())

evaluator_auc = BinaryClassificationEvaluator(
    labelCol='satisfaction', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)

# 9. 5-Fold Cross Validation
cv = CrossValidator(
    estimator=rf_clf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_auc,
    numFolds=5,
    seed=42
)

# 10. Train
cv_model  = cv.fit(train_assembled)
preds_clf = cv_model.transform(test_assembled)

# 11. Evaluation Metrics
accuracy  = MulticlassClassificationEvaluator(labelCol='satisfaction', predictionCol='prediction', metricName='accuracy').evaluate(preds_clf)
f1        = MulticlassClassificationEvaluator(labelCol='satisfaction', predictionCol='prediction', metricName='f1').evaluate(preds_clf)
precision = MulticlassClassificationEvaluator(labelCol='satisfaction', predictionCol='prediction', metricName='weightedPrecision').evaluate(preds_clf)
recall    = MulticlassClassificationEvaluator(labelCol='satisfaction', predictionCol='prediction', metricName='weightedRecall').evaluate(preds_clf)
auc       = evaluator_auc.evaluate(preds_clf)

print(f'Accuracy  : {accuracy*100:.2f}%')
print(f'F1        : {f1:.4f}')
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'AUC-ROC   : {auc:.4f}')

best_rf_model = cv_model.bestModel
print(f"Best Parameters -> Num Trees: {best_rf_model.getNumTrees}, Max Depth: {best_rf_model.getOrDefault('maxDepth')}")

importances = list(zip(feature_columns, best_rf_model.featureImportances.toArray()))
importances_sorted = sorted(importances, key=lambda x: x[1], reverse=True)

Accuracy  : 74.40%
F1        : 0.7470
Precision : 0.7502
Recall    : 0.7440
AUC-ROC   : 0.6618
Best Parameters -> Num Trees: 100, Max Depth: 10


In [ ]:
# Horizontal bar chart of feature importances from the Tuned Random Forest
# Higher importance = feature contributes more to splitting decisions across all trees
feat_df    = pd.DataFrame(importances_sorted, columns=['Feature', 'Importance']).sort_values('Importance')
bar_colors = plt.cm.Blues(np.linspace(0.35, 0.9, len(feat_df)))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(feat_df['Feature'], feat_df['Importance'], color=bar_colors, edgecolor='white')
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importance — RF Classifier (ADASYN & Tuned)')
for bar, val in zip(bars, feat_df['Importance']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('viz3_feature_importance.png', bbox_inches='tight')
plt.show()

delivery_error and delivery_days are the dominant predictors by a big margin compared to the others. Freight cost and total order value contribute comparatively little. This means logistics are the most important to improve customer satisfaction, not product selection or pricing.

## 10. Clustering — Transaction-based Behavioral Grouping (K-Means)

In [ ]:
# Aggregate order-level data up to the customer level
# Each customer gets a single row summarising their full purchase history
df_customer = df_clean.groupBy('customer_id').agg(
    F.count('order_id').alias('num_orders'),
    F.sum('total_order_value').alias('total_spend'),
    F.avg('total_order_value').alias('avg_order_value'),
    F.avg('delivery_days').alias('avg_delivery_days'),
    F.avg('label').alias('avg_review_score'),
    F.avg('freight_value').alias('avg_freight')
).dropna()

print(f'Unique customers: {df_customer.count():,}')



cluster_features = ['num_orders', 'total_spend', 'avg_order_value', 'avg_delivery_days', 'avg_review_score']

# Use StandardScaler because K-Means uses Euclidean distance,
# so features on larger scales (e.g. total_spend in hundreds) would dominate
# features on smaller scales (e.g. num_orders) without normalisation
assembler_k = VectorAssembler(inputCols=cluster_features, outputCol='features_raw', handleInvalid='skip')
scaler_k    = StandardScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)
pipe_k      = Pipeline(stages=[assembler_k, scaler_k])
df_scaled   = pipe_k.fit(df_customer).transform(df_customer)

# Silhouette score measures cluster cohesion, ranges from -1 to 1, higher is better
# Evaluate K=2 to 11 and pick the value with the highest score
evaluator_sil = ClusteringEvaluator(featuresCol='features', metricName='silhouette')
k_scores = {}
for k in range(2,11):
    km   = KMeans(k=k, featuresCol='features', seed=42, maxIter=50)
    pred = km.fit(df_scaled).transform(df_scaled)
    sil  = evaluator_sil.evaluate(pred)
    k_scores[k] = sil
    print(f'K={k}  Silhouette={sil:.4f}')

best_k = max(k_scores, key=k_scores.get)
print(f'Best K={best_k}  (Silhouette={k_scores[best_k]:.4f})')

# Train final model with the best K
model_km     = KMeans(k=best_k, featuresCol='features', seed=42, maxIter=100).fit(df_scaled)
df_clustered = model_km.transform(df_scaled)

# Summary statistics per cluster to use for assigning business labels below
print('Cluster profiles:')
df_clustered.groupBy('prediction').agg(
    F.count('customer_id').alias('customers'),
    F.round(F.avg('total_spend'), 2).alias('avg_spend'),
    F.round(F.avg('num_orders'), 2).alias('avg_orders'),
    F.round(F.avg('avg_review_score'), 2).alias('avg_review'),
    F.round(F.avg('avg_delivery_days'), 1).alias('avg_delivery_days')
).orderBy('prediction').show()

In [ ]:
# Visualise the best clustering result (K=2)
# The 5 scaled features are projected down to 2 principal components with PCA
# so the two clusters can be shown clearly on a single 2D scatter plot.
from pyspark.ml.feature import PCA

pca       = PCA(k=2, inputCol='features', outputCol='pca_features')
pca_model = pca.fit(df_clustered)
df_pca    = pca_model.transform(df_clustered)

plot_pd = df_pca.select('pca_features', 'prediction').toPandas()
plot_pd['pc1'] = plot_pd['pca_features'].apply(lambda v: float(v[0]))
plot_pd['pc2'] = plot_pd['pca_features'].apply(lambda v: float(v[1]))

# Project the cluster centroids into the same PCA space.
# Features are already mean-centred (StandardScaler withMean=True), so a plain
# matrix projection lines the centroids up with the scattered points.
centers     = np.array(model_km.clusterCenters())
centers_pca = centers @ pca_model.pc.toArray()

seg_names = {0: 'Occasional Buyers', 1: 'High-Value Buyers'}

fig, ax = plt.subplots(figsize=(10, 7))
for cl in sorted(plot_pd['prediction'].unique()):
    sub = plot_pd[plot_pd['prediction'] == cl]
    ax.scatter(sub['pc1'], sub['pc2'], s=12, alpha=0.35,
               color=COLORS[cl], label=f'Cluster {cl} - {seg_names.get(cl, "")}')

ax.scatter(centers_pca[:, 0], centers_pca[:, 1], marker='X', s=280,
           color='black', edgecolor='white', linewidth=1.5, label='Centroids', zorder=5)

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title(f'K-Means Clustering Result (K={best_k}, Silhouette={k_scores[best_k]:.3f})',
             fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('viz5_cluster_scatter.png', bbox_inches='tight')
plt.show()

## 11. Visualisations

In [ ]:
# Chart 1: Distribution of raw review scores and broad satisfaction groupings
review_dist = df_clean.groupBy('label').count().orderBy('label').toPandas()
total = review_dist['count'].sum()
review_dist['pct'] = review_dist['count'] / total * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Customer Review Score Distribution', fontsize=15, fontweight='bold')

bar_colors = ['#EF5350', '#FF7043', '#FFA726', '#66BB6A', '#42A5F5']
axes[0].bar(review_dist['label'], review_dist['pct'], color=bar_colors, edgecolor='white')
axes[0].set_xlabel('Review Score (1-5)')
axes[0].set_ylabel('% of Orders')
axes[0].set_title('Score Frequency')
for _, row in review_dist.iterrows():
    axes[0].text(row['label'], row['pct'] + 0.4, f"{row['pct']:.1f}%", ha='center', fontsize=10)

satisfied    = review_dist[review_dist['label'] >= 4]['count'].sum()
neutral      = review_dist[review_dist['label'] == 3]['count'].sum()
dissatisfied = review_dist[review_dist['label'] <= 2]['count'].sum()
axes[1].pie(
    [satisfied, neutral, dissatisfied],
    labels=[
        f'Satisfied (4-5 stars)\n{satisfied/total*100:.1f}%',
        f'Neutral (3 stars)\n{neutral/total*100:.1f}%',
        f'Dissatisfied (1-2 stars)\n{dissatisfied/total*100:.1f}%'
    ],
    colors=['#42A5F5', '#FFA726', '#EF5350'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Satisfaction Breakdown')
plt.tight_layout()
plt.savefig('viz1_review_distribution.png', bbox_inches='tight')
plt.show()


The majority of customers leave 4- or 5-star reviews. The dissatisfied segment (1-2 stars) is small in proportion but disproportionately important for retention: a single bad experience can eliminate repeat purchases entirely. Reducing that tail is a higher-value intervention than further optimising the already-positive majority.

In [ ]:
# Chart 2: Average delivery time and delivery error (lateness) broken down by review score.
# Shows whether slower or less accurate deliveries correlate with lower scores.
delivery_pd = df_clean.groupBy('label').agg(
    F.round(F.avg('delivery_days'), 1).alias('avg_delivery_days'),
    F.round(F.avg('delivery_error'), 1).alias('avg_delay')
).orderBy('label').toPandas()

x     = delivery_pd['label']
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
fig.suptitle('Delivery Performance vs Review Score', fontsize=14, fontweight='bold')

b1 = ax.bar(x - width/2, delivery_pd['avg_delivery_days'], width, label='Avg Delivery Days',    color='#2196F3', alpha=0.85)
b2 = ax.bar(x + width/2, delivery_pd['avg_delay'],         width, label='Avg Delay vs Estimate', color='#FF5722', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')  # zero line for delay reference
ax.set_xlabel('Review Score')
ax.set_ylabel('Days')
ax.legend()
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.2, f'{h:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('viz2_delivery_vs_review.png', bbox_inches='tight')
plt.show()


1-star orders show both the longest delivery times and the largest positive delay (i.e. arriving later than promised). The relationship is monotonic where every star improvement corresponds to measurably shorter and more punctual deliveries.

In [ ]:
#Chart 3: Transaction-Based Behavioral Grouping
cluster_pd = df_clustered.groupBy('prediction').agg(
    F.count('customer_id').alias('customers'),
    F.round(F.avg('total_spend'), 0).alias('avg_spend'),
    F.round(F.avg('avg_delivery_days'), 1).alias('avg_delivery_days'),
    F.round(F.avg('avg_freight'), 0).alias('avg_freight'),
).orderBy('avg_spend').toPandas()

seg_labels = ['Occasional Buyers', 'High-Value Buyers'][:len(cluster_pd)]
cluster_pd['segment'] = seg_labels
seg_colors = COLORS[:len(cluster_pd)]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(f'Transaction-Based Behavioral Grouping (K={best_k})', fontsize=14, fontweight='bold')

for ax, col_name, title, prefix, fmt in zip(
    axes,
    ['customers',    'avg_spend',  'avg_delivery_days',  'avg_freight'],
    ['Customers',    'Avg Spend',  'Avg Delivery Days',  'Avg Freight'],
    ['',             '$ ',        '',                   '$ '],
    ['{:,.0f}',      '{:,.0f}',    '{:.1f}',             '{:,.0f}']
):
    bars = ax.bar(cluster_pd['segment'], cluster_pd[col_name], color=seg_colors, width=0.5)
    ax.set_title(title, fontsize=12)
    ax.tick_params(axis='x', rotation=15)
    ax.spines[['top', 'right']].set_visible(False)
    for i, v in enumerate(cluster_pd[col_name]):
        ax.text(i, v * 1.02, prefix + fmt.format(v), ha='center', fontsize=10, fontweight='bold')
    ax.set_xlim(-0.5, len(cluster_pd) - 0.5)
    ax.set_ylim(0, cluster_pd[col_name].max() * 1.25)

plt.tight_layout()
plt.savefig('viz4_behavioral_grouping.png', bbox_inches='tight')
plt.show()

High-value buyers spend nearly 8x more ($ 1,007 vs $ 129) and experience slightly longer delivery times (14.9 vs 12.4 days), likely due to heavier items reflected in higher freight costs ($ 49 vs $ 19). The vast majority of customers (96.5%) fall in the occasional buyer segment, making high-value buyers a small but significantly different group.